1. Set Up & Installation

In [16]:
import torch
import torch.nn as nn
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.preprocess import quant_pre_process

2. Define Model Architecture

In [20]:
class VideoClassifierLite(nn.Module):
    def __init__(self, num_classes=2):
        super(VideoClassifierLite, self).__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Reduces 112x112 to 56x56
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # THE FIX: Standard AvgPool instead of Adaptive. ONNX loves this.
            # 56x56 input -> kernel 20, stride 18 -> exactly 3x3 output
            nn.AvgPool2d(kernel_size=20, stride=18), 
            nn.Flatten()
        )
        
        self.lstm = nn.LSTM(input_size=288, hidden_size=32, num_layers=1, batch_first=True)
        
        self.fc = nn.Sequential(
            nn.Dropout(0.7), 
            nn.Linear(32 * 2, num_classes) 
        )

    def forward(self, x):
        b, t, c, h, w = x.size()
        x = x.view(b * t, c, h, w)
        x = self.cnn(x)
        x = x.view(b, t, -1)
        
        lstm_out, _ = self.lstm(x)
        
        first_feat = lstm_out[:, 0, :]
        last_feat = lstm_out[:, -1, :]
        combined = torch.cat((first_feat, last_feat), dim=1)
        
        return self.fc(combined)

3. Load

In [21]:
model = VideoClassifierLite()
model_path = r"D:\Skripsi_Raphaela\conv_lstm\program\best_model_convlstm.pth" 

print("Loading weights...")
model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True), strict=True)
model.eval()
print("Weights loaded successfully!")

Loading weights...
Weights loaded successfully!


5. Export to ONNX (FP32)

In [22]:
print("Converting to ONNX format...")
onnx_model_path = r"D:\Skripsi_Raphaela\conv_lstm\program\convlstm_fp32.onnx"

SEQ_LEN = 15 
dummy_input = torch.randn(1, SEQ_LEN, 3, 112, 112)

torch.onnx.export(
    model, dummy_input, onnx_model_path,
    export_params=True, opset_version=14, do_constant_folding=True,
    input_names=['input_frames'], output_names=['output_class'],
    dynamic_axes={'input_frames': {0: 'batch_size'}, 'output_class': {0: 'batch_size'}}
)

Converting to ONNX format...


d:\Skripsi_Raphaela\rt_detr\program\.venv\Lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


6. Quantize to ONNX (INT8)

In [23]:
preprocessed_model_path = r"D:\Skripsi_Raphaela\conv_lstm\program\convlstm_fp32_prep.onnx"
quantized_model_path = r"D:\Skripsi_Raphaela\conv_lstm\program\convlstm_int8.onnx"

quant_pre_process(input_model_path=onnx_model_path, output_model_path=preprocessed_model_path, skip_optimization=False)

print("Quantization to INT8...")
quantize_dynamic(
    model_input=preprocessed_model_path,
    model_output=quantized_model_path,
    weight_type=QuantType.QInt8
)

print(f"[DONE] Model INT8 saved in : {quantized_model_path}")

Quantization to INT8...
[DONE] Model INT8 saved in : D:\Skripsi_Raphaela\conv_lstm\program\convlstm_int8.onnx
